In [ ]:
# ── Step 1: Clean up old processes ─────────────────────────────────────────
import subprocess
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
print('✅ Cleaned up old processes')

# ── Step 2: Install ComfyUI ────────────────────────────────────────────────
import os
if not os.path.exists('/kaggle/working/ComfyUI'):
    print('Installing ComfyUI...')
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI
    %cd /kaggle/working/ComfyUI
    !pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
else:
    print('✅ ComfyUI already installed')
    %cd /kaggle/working/ComfyUI

# ── Step 3: Download Models (skip if exist) ──────────────────────────────
models = [
    ('models/checkpoints/sd_xl_base_1.0.safetensors',
     'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors'),
    ('models/upscale_models/RealESRGAN_x4plus.pth',
     'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'),
    ('models/vae/sdxl_vae.safetensors',
     'https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors'),
]
for fpath, url in models:
    if not os.path.exists(fpath):
        print(f'Downloading {os.path.basename(fpath)}...')
        !wget -q -O {fpath} {url}
    else:
        print(f'✅ {os.path.basename(fpath)} already exists')

# ── Step 4: Install Cloudflared ────────────────────────────────────────────
if not os.path.exists('/kaggle/working/cloudflared'):
    !wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
    !chmod +x /kaggle/working/cloudflared
print('✅ Cloudflared ready')

# ── Step 5: Cloudflare tunnel + URL reporter ───────────────────────────────
import re, threading, time, socket
from IPython.display import display, HTML

def iframe_thread(port):
    for _ in range(120):
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0:
            sock.close()
            break
        sock.close()
    print('\nComfyUI is running. Starting Cloudflare tunnel...')

    p = subprocess.Popen(
        ['/kaggle/working/cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in p.stdout:
        l = line.decode(errors='ignore')
        m = re.search(r'https?://[a-zA-Z0-9-]+\.trycloudflare\.com', l)
        if m:
            url = m.group(0)
            print('\n=======================================================')
            print(f'\u2705 YOUR SERVER URL: {url}')
            display(HTML(f'<a href="{url}" target="_blank" style="font-size:1.2em;font-weight:bold">{url}</a>'))
            print('=======================================================\n')
            break

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# ── Step 6: Launch ComfyUI ─────────────────────────────────────────────────
print('\n🚀 Starting ComfyUI...')
!python main.py --dont-print-server --enable-cors-header "*"